# c5_90—Publish snapshot to Cloud Storage

**ROI maintainers only. This is not for students.** Running it as an attendee throws a permissions
error on the upload, which is expected—you do not have write access to our bucket.

It exists to kill a single point of failure. `c5_01_load_explore.ipynb` pulls live from CDC's
Socrata endpoints, HRSA's ArcGIS service and CMS's ArcGIS service. If any one of them is down,
slow, or rate-limiting at 9am on event day, it fails for the whole room at once—and this challenge
leans harder on ArcGIS than any other in the pack, on a service we have already watched return
`400 Unable to complete operation` and then succeed on the retry. `scripts/load.sh` rebuilds the
identical tables from a Cloud Storage snapshot instead, and this notebook is what produces it.

## The rule that makes it trustworthy

**This notebook does not reimplement the student notebook.** It fetches it by raw URL, executes it
once per county with `COUNTY_FIPS`, `COUNTY_NAME`, `STATE_ABBR` and `DATASET` overridden, and
exports what lands in BigQuery. That guarantees the fallback cannot drift from what students
actually get—which it silently would, the first time somebody fixed a bug in one file and not the
other.

Four design choices worth knowing about, three of them inherited from Challenge 4 and one specific
to this challenge:

- **We export from BigQuery, not from dataframes in memory.** `care_sites` is built by a
  `CREATE OR REPLACE TABLE` that spatially joins the in-memory frame against county boundaries, so
  the frame is missing two columns the finished table has. Reading back from BigQuery means the
  snapshot is by construction the notebook's end state.
- **We gate on the student notebook's own validation**, read back out of the executed namespace,
  and refuse to publish a county whose checks did not run at all. A suite reporting "no failures"
  because it never executed is the most dangerous green there is.
- **We export with `extract_table`, not `df.to_parquet`.** BigQuery writes the real schema.
  Pandas infers one, and on this challenge that difference is load-bearing: `shortage_areas` is
  legitimately **empty** in some counties, and an empty dataframe round-tripped through pandas
  produces a parquet file with no usable column types.
- **We run every cell, including the Section 2 hook.** Challenge 4 skipped its hook cells because
  each one spent a Grounding with Google Maps call and eight states of those was real money.
  Section 2 here costs about 36 seconds of CDC's compute and nothing else, so skipping it would
  buy a few minutes and give up the strongest claim this notebook makes—that the snapshot is what
  you get from executing the student notebook, all of it, unmodified except for the county.

In [ ]:
BUCKET       = "class-demo"
PREFIX       = "a4i-2026/challenge-5-public-health"
NOTEBOOK_URL = ("https://raw.githubusercontent.com/haggman/"
                "A4I2026-challenge-5-public-health/main/notebooks/c5_01_load_explore.ipynb")

# Never publish out of the dataset a human has been poking at. The student notebook's
# own §9 inventory reads __TABLES__, and we have already seen a reused project carrying
# _tmp_sites and _tmp_tracts left over from an older version of the notebook.
SCRATCH_DATASET = "a4i_c5_publish_scratch"

# The counties we pre-stage. STATE_ABBR is written out by hand because it must be right
# and it is human-checkable at a glance; the FIPS-to-name pairing is NOT taken on trust,
# it is resolved against bigquery-public-data.geo_us_boundaries.counties in the next cell.
#
# Toronto has no entry here on purpose. Every dataset in this challenge is US federal, so
# Toronto attendees pick a US county like everybody else, and Fulton is the default.
COUNTIES = [
    ("13121", "Fulton",      "GA"),   # Atlanta — the notebook default
    ("17031", "Cook",        "IL"),   # Chicago
    ("06085", "Santa Clara", "CA"),   # Sunnyvale
    ("36005", "Bronx",       "NY"),   # New York City
    ("48201", "Harris",      "TX"),   # Houston — measured during research
    ("48453", "Travis",      "TX"),   # Austin  — measured during research
]

# Every table the student notebook creates, named explicitly. NOT a __TABLES__ listing:
# that would sweep up anything else living in the dataset.
TABLES = ["burden_tracts", "exposure_tracts", "care_sites", "care_sites_state",
          "shortage_areas", "shortage_facilities", "disease_weekly", "tract_access"]

# Tables where zero rows means the load is broken. `shortage_areas` and
# `shortage_facilities` are deliberately NOT on this list: Fulton, Harris and Travis
# all genuinely have zero live HPSA shortage areas, we measured it, and that is the
# finding rather than a fault — Fulton is the notebook default and ships empty. They still get published as empty tables so load.sh has something to load and
# the student's queries find the table they expect.
REQUIRED_NONEMPTY = ["burden_tracts", "exposure_tracts", "care_sites",
                     "care_sites_state", "tract_access", "disease_weekly"]

# The student notebook records 33 checks today. Demand most of them rather than an exact
# count, so adding one does not break publishing but deleting a section does.
MIN_CHECKS = 24

# The harness in c5_01 keys its verdicts under "verdict". Challenge 4's used "result".
# This constant exists so that fact is stated once, loudly, in the place that depends on
# it. See the gate cell for why getting it wrong is worse than it sounds.
VERDICT_KEY = "verdict"

import os, subprocess
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True).strip()

print(f"project  : {PROJECT_ID}")
print(f"target   : gs://{BUCKET}/{PREFIX}/<COUNTY_FIPS>/<table>/data.parquet")
print(f"counties : {', '.join(f'{n} {f}' for f, n, _ in COUNTIES)}")
print(f"tables   : {len(TABLES)}  ({len(REQUIRED_NONEMPTY)} of them must be non-empty)")

## Resolve the counties before running anything

A wrong FIPS code does not fail. It produces a complete, plausible, internally consistent snapshot
of the wrong place, and nothing downstream can tell. `13121` and `13221` are both real Georgia
counties and only one of them is Atlanta.

So the codes are not trusted. Each one is resolved against
`bigquery-public-data.geo_us_boundaries.counties`—the same table the student notebook already
uses for its spatial join, so it is verified by the fact that Section 7 works—and the name it
returns has to match the name written in the config cell. A mismatch stops the run here, before
fifteen minutes have been spent.

**`STATE_ABBR` is checked differently, and it is worth knowing how.** There is no state-abbreviation
column we have verified on that table, so rather than guess at one, the abbreviation stays
hand-written and gets caught by the data: Section 7 pulls facilities for `STATE_ABBR` and then
spatially joins them into the county. Give it the wrong state and the join produces zero care
sites, which trips the student notebook's own `at least one care site in the county` check, which
trips the gate. The failure is loud and it lands in the right place.

In [ ]:
import pandas as pd
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID, location="US")

fips_list = ", ".join(f"'{f}'" for f, _, _ in COUNTIES)
resolved = bq.query(f"""
    SELECT geo_id, county_name
    FROM `bigquery-public-data.geo_us_boundaries.counties`
    WHERE geo_id IN ({fips_list})
""").to_dataframe()
lookup = dict(zip(resolved["geo_id"], resolved["county_name"]))

problems = []
for fips, name, state in COUNTIES:
    if fips not in lookup:
        problems.append(f"{fips} ({name}, {state}) does not exist in geo_us_boundaries.counties")
    elif lookup[fips].lower() != name.lower():
        problems.append(f"{fips} is {lookup[fips]!r}, not {name!r} — one of them is wrong")
    else:
        print(f"  {fips}  {lookup[fips]:<14} {state}   confirmed")

if len(set(f for f, _, _ in COUNTIES)) != len(COUNTIES):
    problems.append("the same FIPS code appears twice in COUNTIES")

if problems:
    for p in problems:
        print(f"  PROBLEM: {p}")
    raise RuntimeError(f"{len(problems)} county problem(s). Fix COUNTIES before publishing — "
                       f"a wrong FIPS produces a perfectly valid snapshot of the wrong place.")
print(f"\nAll {len(COUNTIES)} counties resolve to the names claimed for them.")

## Fetch the student notebook and rewrite its config cell

Every code cell runs. Nothing is skipped, so there is no skip-marker machinery to rot.

What does need care is the config cell. We locate it **and prove it is unambiguous** before
touching it. The regexes are the part most likely to break: a rename in the student notebook that
stops a pattern matching would make `re.subn` substitute nothing, report nothing, and publish six
identical copies of Cook County. So every substitution count is asserted, and all four overrides
are test-fired on a probe before the long run starts.

Note the two lines that must **not** match. The config cell also contains

```
STATE_FIPS = COUNTY_FIPS[:2]
COUNTY_ONLY = COUNTY_FIPS[2:]
```

and both of those have to survive intact, because everything downstream is derived from them.
Anchoring each pattern at the start of a line with the full variable name handles it, and the
assertions below prove it rather than assuming it.

In [ ]:
import re
import requests
import nbformat

resp = requests.get(NOTEBOOK_URL, timeout=60)
resp.raise_for_status()
nb = nbformat.reads(resp.text, as_version=4)

sources = []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    if cell.source.lstrip().startswith("%%"):     # exec() cannot run cell magics
        raise RuntimeError("the student notebook has gained a cell magic; exec() cannot "
                           "run it and this publisher would silently skip it")
    sources.append(cell.source)

print(f"fetched {len(nb.cells)} cells, {len(sources)} of them executable code")

# --- Find the config cell, once, and prove it is unambiguous ----------------
PATTERNS = {
    "COUNTY_FIPS": re.compile(r'(?m)^COUNTY_FIPS\s*=\s*.+$'),
    "COUNTY_NAME": re.compile(r'(?m)^COUNTY_NAME\s*=\s*.+$'),
    "STATE_ABBR":  re.compile(r'(?m)^STATE_ABBR\s*=\s*.+$'),
    "DATASET":     re.compile(r'(?m)^DATASET\s*=\s*.+$'),
}

config_idx = [i for i, s in enumerate(sources)
              if all(p.search(s) for p in PATTERNS.values())]
if len(config_idx) != 1:
    raise RuntimeError(
        f"Expected exactly one config cell assigning all of {list(PATTERNS)}; found "
        f"{len(config_idx)}: {config_idx}. The student notebook's config cell has changed "
        f"shape and the overrides below would not work. Fix this before publishing anything.")
CONFIG_IDX = config_idx[0]

# The two derived lines that must survive. STATE_FIPS starts with a different word than
# STATE_ABBR, and COUNTY_ONLY than COUNTY_NAME, so line-anchored full-name patterns
# exclude them by construction — proven here, not assumed.
assert not PATTERNS["STATE_ABBR"].search("STATE_FIPS = COUNTY_FIPS[:2]"), \
    "STATE_ABBR pattern is too loose — it would rewrite STATE_FIPS"
assert not PATTERNS["COUNTY_NAME"].search("COUNTY_ONLY = COUNTY_FIPS[2:]"), \
    "COUNTY_NAME pattern is too loose — it would rewrite COUNTY_ONLY"


def configure(src, fips, name, state):
    """Rewrite the four config lines. Raises rather than no-ops."""
    values = {"COUNTY_FIPS": f'COUNTY_FIPS = "{fips}"',
              "COUNTY_NAME": f'COUNTY_NAME = "{name}"',
              "STATE_ABBR":  f'STATE_ABBR = "{state}"',
              "DATASET":     f'DATASET = "{SCRATCH_DATASET}"'}
    for key, pat in PATTERNS.items():
        src, n = pat.subn(lambda _m: values[key], src, count=1)
        if n != 1:
            raise RuntimeError(f"{key} substitution matched {n} lines, expected 1")
    return src


# Prove all four work before spending fifteen minutes finding out they do not.
probe = configure(sources[CONFIG_IDX], "48201", "Harris", "TX")
for expect in ('COUNTY_FIPS = "48201"', 'COUNTY_NAME = "Harris"', 'STATE_ABBR = "TX"',
               f'DATASET = "{SCRATCH_DATASET}"'):
    assert expect in probe, f"override did not take: {expect}"
assert "STATE_FIPS = COUNTY_FIPS[:2]" in probe, "the STATE_FIPS derivation was clobbered"
assert "COUNTY_ONLY = COUNTY_FIPS[2:]" in probe, "the COUNTY_ONLY derivation was clobbered"
print(f"Config cell is #{CONFIG_IDX}; all four overrides test-fired successfully.")

## Test the gate before trusting it

The gate is the only thing standing between a broken run and a snapshot that 150 people load on
event day, so it gets tested rather than assumed. It is handed synthetic namespaces that *should*
be rejected, and each one has to be.

**One of those negative controls is not hypothetical.** Challenge 4's harness records verdicts
under the key `result`. This challenge's records them under `verdict`. Copy Challenge 4's gate
across unchanged and `ck.get("result") == "FAIL"` is `False` for every check ever recorded—so the
gate finds no failures, announces success, and publishes whatever came out. **A gate that reads the
wrong key does not fail closed. It fails open, silently, and looks exactly like a clean run.**

So this gate does not merely read `verdict`. It asserts that every check dict actually contains
that key, which turns the whole class of mistake from an invisible pass into a loud stop.

Then two positive controls: a clean run, and a run carrying WARNs. The second matters. The student
notebook distinguishes **our load being broken** (`check`, FAIL) from **the publisher's data being
untidy** (`note`, WARN), and Fulton County produces five WARNs on a perfectly good run—the doubled
PM2.5 rows, the file whose title claims two years and contains one, the HPSA designations marked
proposed-for-withdrawal, the handful of tracts EPA models and PLACES suppresses, and the county
having no live shortage-area designation at all. All three are teaching material. A gate that treated them as
disqualifying would publish nothing at all and look like it was working.

In [ ]:
def gate(ns, fips, name):
    """Raise if this run must not be published. Every branch here is tested below."""
    ran_as = ns.get("COUNTY_FIPS")
    if ran_as != fips:
        raise RuntimeError(
            f"county override failed: asked for {fips!r}, notebook ran as {ran_as!r}. "
            f"Nothing uploaded. Fix configure() before rerunning.")

    if ns.get("DATASET") != SCRATCH_DATASET:
        raise RuntimeError(
            f"{name} validated against DATASET={ns.get('DATASET')!r}, not "
            f"{SCRATCH_DATASET!r}. Those checks graded the wrong tables.")

    checks = ns.get("CHECKS")
    if not checks:
        raise RuntimeError(
            "the validation section produced no checks — it did not run. Refusing to "
            "publish something nobody checked.")
    if len(checks) < MIN_CHECKS:
        raise RuntimeError(
            f"only {len(checks)} checks ran, expected at least {MIN_CHECKS}. The validation "
            f"section is incomplete.")

    # The shape assertion. Without it, a renamed verdict key makes every comparison below
    # False and this function becomes an expensive way of returning None.
    shapeless = [ck for ck in checks if VERDICT_KEY not in ck]
    if shapeless:
        raise RuntimeError(
            f"{len(shapeless)} of {len(checks)} checks carry no {VERDICT_KEY!r} key — e.g. "
            f"{shapeless[0]}. The harness in c5_01 has been renamed and this gate is now "
            f"reading a key that does not exist, which means it cannot see a failure. "
            f"Fix VERDICT_KEY before publishing anything.")

    failed = [ck for ck in checks if ck[VERDICT_KEY] == "FAIL"]
    if failed:
        raise RuntimeError("validation failed: " +
                           "; ".join(f"{ck['check']} ({ck['detail']})" for ck in failed))
    warned = [ck for ck in checks if ck[VERDICT_KEY] == "WARN"]
    if warned:
        print(f"  {len(warned)} WARN (source data, not our load): " +
              "; ".join(ck["check"] for ck in warned))
    return len(checks), len(warned)


def _ns(fips="17031", dataset=None, checks=None):
    return {"COUNTY_FIPS": fips, "DATASET": dataset or SCRATCH_DATASET, "CHECKS": checks}


_ok  = [{"check": f"c{i}", VERDICT_KEY: "PASS", "detail": ""} for i in range(MIN_CHECKS)]
_bad = _ok[:-1] + [{"check": "boom", VERDICT_KEY: "FAIL", "detail": "on purpose"}]
# Challenge 4's shape. This is the control that matters most.
_c4  = [{"check": f"c{i}", "result": "PASS", "detail": ""} for i in range(MIN_CHECKS)]

NEGATIVE_CONTROLS = {
    "wrong county":       (_ns(fips="48201"), "17031", "Cook"),
    "wrong dataset":      (_ns(dataset="a4i_health"), "17031", "Cook"),
    "no checks at all":   (_ns(checks=None), "17031", "Cook"),
    "too few checks":     (_ns(checks=_ok[:3]), "17031", "Cook"),
    "a failing check":    (_ns(checks=_bad), "17031", "Cook"),
    "C4-shaped verdicts": (_ns(checks=_c4), "17031", "Cook"),
}

for label, (ns, fips, name) in NEGATIVE_CONTROLS.items():
    try:
        gate(ns, fips, name)
    except RuntimeError as exc:
        print(f"  rejected as it should be — {label:<20} {str(exc)[:58]}")
    else:
        raise RuntimeError(
            f"THE GATE IS BROKEN: {label!r} was accepted. Do not publish anything from this "
            f"run until it is fixed — a gate that passes everything is worse than no gate.")

_warn = _ok[:-1] + [{"check": "untidy source", VERDICT_KEY: "WARN", "detail": "doubled rows"}]
gate(_ns(checks=_ok), "17031", "Cook")
gate(_ns(checks=_warn), "17031", "Cook")
print(f"\nGate accepts a clean run and a run carrying WARNs, and rejects all "
      f"{len(NEGATIVE_CONTROLS)} negative controls.")

## Build, validate, then export—one county at a time

Each county runs the whole student notebook end to end into the scratch dataset, passes the gate,
gets two assertions made against the **data** rather than the config, and is then exported straight
out of BigQuery.

The data assertions are there because row counts cannot tell six counties apart. A publish loop
with a broken override produces six perfectly valid, perfectly plausible snapshots of Chicago. So:
every tract ID must start with this county's FIPS, and every care site must have been assigned to
this county by the spatial join.

A failure on one county does not stop the others. The summary at the bottom is what you read.

Budget roughly **two and a half minutes per county**—about 130 seconds for the notebook plus the
exports—so six counties is somewhere around a quarter of an hour.

In [ ]:
import time, traceback
from google.cloud import storage

# The student notebook calls display() in five cells. exec() runs in a namespace we build
# by hand, and that namespace does not inherit the notebook's injected globals — so
# display resolves to nothing and every table cell raises NameError. Pass it in.
try:
    from IPython.display import display as _display
except Exception:                                   # not in a notebook at all
    _display = print

NOTEBOOK_BUILTINS = {"display": _display}

gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

results = {}

for fips, name, state in COUNTIES:
    print(f"\n{'=' * 68}\n{name} County, {state}  ({fips})\n{'=' * 68}")
    t0 = time.time()
    ns = {"__name__": "__main__", **NOTEBOOK_BUILTINS}
    try:
        # Start every county from an empty dataset. Not paranoia — the first publish run
        # showed Fulton's Section 9 inventory reporting tract_access = 1,328 rows, which
        # is COOK's number, because Section 9 inventories the dataset before Section 10
        # rebuilds that table. Harmless there, since every table is rewritten before the
        # export. But a county that dies halfway leaves a full set of the previous
        # county's tables sitting in the dataset, and the only thing standing between
        # that and a mislabelled snapshot is that nothing currently skips a section.
        # That is a property of today's notebook, not a guarantee, and it is one line
        # to stop depending on it.
        bq.delete_dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}",
                          delete_contents=True, not_found_ok=True)

        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, fips, name, state)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

        n_checks, n_warn = gate(ns, fips, name)

        # Does the DATA say it is this county, or only the config?
        S = f"{PROJECT_ID}.{SCRATCH_DATASET}"
        row = list(bq.query(f"""
            SELECT (SELECT COUNT(*) FROM `{S}.burden_tracts`) AS n_tracts,
                   (SELECT COUNTIF(SUBSTR(geo_id, 1, 5) = '{fips}')
                      FROM `{S}.burden_tracts`) AS right_county,
                   (SELECT COUNT(*) FROM `{S}.care_sites`) AS n_sites,
                   (SELECT COUNTIF(county_geo_id = '{fips}')
                      FROM `{S}.care_sites`) AS right_sites
        """).result())[0]
        if row.right_county != row.n_tracts:
            raise RuntimeError(f"{row.n_tracts - row.right_county} of {row.n_tracts} tracts "
                               f"do not start with FIPS {fips}")
        if row.n_sites == 0:
            raise RuntimeError(f"no care sites landed in {name}. The usual cause is a wrong "
                               f"STATE_ABBR: the facility pull succeeded for the wrong state "
                               f"and the spatial join into the county matched nothing.")
        if row.right_sites != row.n_sites:
            raise RuntimeError(f"{row.n_sites - row.right_sites} care sites carry the wrong "
                               f"county_geo_id")

        # Parquet extract cannot carry a GEOGRAPHY column. Today none of these tables has
        # one — care_sites stores longitude/latitude and the geography is built inline in
        # the queries — but that is a property of the current notebook, not a guarantee.
        geo = bq.query(f"""
            SELECT table_name, column_name
            FROM `{S}`.INFORMATION_SCHEMA.COLUMNS
            WHERE data_type = 'GEOGRAPHY' AND table_name IN
                  ({', '.join(f"'{t}'" for t in TABLES)})
        """).to_dataframe()
        if len(geo):
            raise RuntimeError(
                f"GEOGRAPHY columns present and Parquet extract cannot carry them: "
                f"{list(zip(geo.table_name, geo.column_name))}. Either cast to WKT in the "
                f"student notebook or change the export format here.")

        # --- Export, straight out of BigQuery ---------------------------------
        written, empties = [], []
        for table in TABLES:
            ref = f"{PROJECT_ID}.{SCRATCH_DATASET}.{table}"
            n = bq.get_table(ref).num_rows
            if n == 0 and table in REQUIRED_NONEMPTY:
                raise RuntimeError(f"required table {table} was empty")

            path = f"{PREFIX}/{fips}/{table}/data.parquet"
            job = bq.extract_table(
                ref, f"gs://{BUCKET}/{path}",
                job_config=bigquery.ExtractJobConfig(destination_format="PARQUET"))
            job.result()

            # An extract of a zero-row table is the case we are least sure of, so it is
            # checked rather than assumed. If BigQuery wrote nothing, put an empty frame
            # with the real column names there instead, so load.sh finds a file.
            if not bucket.blob(path).exists():
                if n:
                    raise RuntimeError(f"{table} extracted {n:,} rows but no object appeared "
                                       f"at gs://{BUCKET}/{path}")
                import io
                cols = [f.name for f in bq.get_table(ref).schema]
                buf = io.BytesIO()
                pd.DataFrame(columns=cols).to_parquet(buf, index=False)
                buf.seek(0)
                bucket.blob(path).upload_from_file(
                    buf, content_type="application/octet-stream")
                print(f"    {table}: 0 rows, extract wrote no object — uploaded an empty "
                      f"frame with {len(cols)} columns instead")

            (empties if n == 0 else written).append(f"{table}({n:,})")

        print(f"  uploaded: {', '.join(written)}")
        if empties:
            print(f"  published empty (legitimately): {', '.join(empties)}")
        results[fips] = ("OK", name, f"{row.n_tracts:,} tracts, {row.n_sites:,} sites, "
                                     f"{n_checks} checks / {n_warn} warn, "
                                     f"{time.time() - t0:.0f}s")

    except Exception as exc:                                       # noqa: BLE001
        results[fips] = ("FAILED", name, str(exc)[:300])
        print(f"  FAILED: {exc}")
        traceback.print_exc(limit=3)

print(f"\n{'=' * 68}\nSUMMARY\n{'=' * 68}")
for fips, (status, name, detail) in results.items():
    print(f"{status:<8} {name:<14} {fips}  {detail}")

## Verify the snapshot is loadable—and that no two counties are copies

Row counts cannot tell six counties apart. The checks in the loop above run against BigQuery; these
run against **what actually landed in Cloud Storage**, which is a different claim and the one
`load.sh` depends on.

Each county's `burden_tracts` is read back out of the bucket and fingerprinted on the hash of its
tract-ID set. If two fingerprints match, something published the same county twice.

The read is done by downloading the blob rather than handing a `gs://` URI to pandas, because the
second needs `gcsfs` installed and the first needs nothing that is not already here.

In [ ]:
import hashlib, io

problems, fingerprints, rows = [], {}, []

for fips, name, state in COUNTIES:
    if results.get(fips, ("",))[0] != "OK":
        continue
    try:
        blob = bucket.blob(f"{PREFIX}/{fips}/burden_tracts/data.parquet")
        df = pd.read_parquet(io.BytesIO(blob.download_as_bytes()))

        ids = sorted(str(x) for x in df["geo_id"])
        fp = hashlib.sha256("|".join(ids).encode()).hexdigest()[:16]

        wrong = sum(1 for i in ids if not i.startswith(fips))
        if wrong:
            problems.append(f"{name}: {wrong} tract IDs do not start with {fips}")
        if fp in fingerprints:
            problems.append(f"{name}: identical tract set to {fingerprints[fp]} — "
                            f"the override did not take for one of them")
        fingerprints[fp] = name

        # Two per-county numbers that decide whether the challenge is workable here, as
        # opposed to whether the load worked. Both have surprised us already.
        shortage = pd.read_parquet(io.BytesIO(bucket.blob(
            f"{PREFIX}/{fips}/shortage_areas/data.parquet").download_as_bytes()))
        access = pd.read_parquet(io.BytesIO(bucket.blob(
            f"{PREFIX}/{fips}/tract_access/data.parquet").download_as_bytes()))
        far = int((access["km_to_safety_net"] > 5).sum())

        rows.append({"county": name, "state": state, "fips": fips,
                     "tracts": len(df), "fingerprint": fp,
                     "shortage_areas": len(shortage),
                     "tracts_over_5km_from_safety_net": far})
    except Exception as exc:                                       # noqa: BLE001
        problems.append(f"{name}: could not read back — {exc}")

display(pd.DataFrame(rows))

print("\nA county with 0 shortage areas is a measured fact, not a broken load — Harris and")
print("Travis are both genuinely zero. A county with 0 tracts beyond 5 km from a safety-net")
print("site is different: the access story is the spine of this challenge, and a county")
print("where nobody is far from care will not carry it. Check that column before the event.")

if problems:
    print("\nPROBLEMS")
    for p in problems:
        print(f"  {p}")
    raise RuntimeError(f"{len(problems)} snapshot problem(s) — do not announce this snapshot.")
print("\nEvery published county is distinct and is the county it claims to be.")

## Finally: the summary

Nothing below this line can fail in a way that matters. That is deliberate.

A previous challenge's publisher uploaded and verified ten instances correctly and then threw a
permissions error in its last cell, because it called an API it never needed. A clean publish
looked like a failed run. **A final reporting cell must not be able to fail after the work has
succeeded**, and it must not attempt a check the notebook is structurally unable to perform—a
publisher running as a maintainer cannot prove a student's read access. Only `bash scripts/load.sh`
in a fresh Skills project can do that, and it is a separate step on the Stage 3 checklist.

In [ ]:
print("=" * 70)
print("C5 PUBLISH—SUMMARY")
print("=" * 70)
ok  = [(f, n) for f, (st, n, _) in results.items() if st == "OK"]
bad = [(f, n) for f, (st, n, _) in results.items() if st != "OK"]
print(f"published : {len(ok)}/{len(COUNTIES)}  {', '.join(n for _, n in ok)}")
if bad:
    print(f"FAILED    : {', '.join(f'{n} ({f})' for f, n in bad)}")
print(f"location  : gs://{BUCKET}/{PREFIX}/<COUNTY_FIPS>/<table>/data.parquet")
print()
print("The bucket needs allAuthenticatedUsers:objectViewer—NOT allUsers.")
print("load.sh never makes an HTTP request: it uses `gcloud storage` and `bq load`")
print("against gs:// URIs, and a BigQuery load job reads the source object as the")
print("job submitter. Every attendee is signed in inside the lab, so authenticated")
print("access covers them and the bucket does not need opening to the world.")
print()
print("Students pass a COUNTY FIPS to load.sh, not a state:")
print("    bash scripts/load.sh 17031")
print("That is a deliberate difference from Challenge 4 and it needs saying in the")
print("README, because five digits typed as a bare number lose the leading zero and")
print("Santa Clara becomes 6085, which matches nothing.")
print()
print("NEXT, and it is the only thing that proves any of this worked:")
print("  run `bash scripts/load.sh 17031` in a FRESH Skills project, then run it again.")
print("  The second run is the one that matters—it takes the other branch.")
print("=" * 70)

try:
    for fips, name in ok:
        blobs = list(gcs.list_blobs(BUCKET, prefix=f"{PREFIX}/{fips}/"))
        size = sum(b.size or 0 for b in blobs) / 1e6
        print(f"  {name:<14} {fips}  {len(blobs):>2} objects, {size:6.1f} MB")
except Exception as exc:                                           # noqa: BLE001
    print(f"  (object listing unavailable: {type(exc).__name__} — the upload above still "
          f"succeeded, this line is cosmetic)")